# Linear Algebra for Neural Data Representation: Interactive Lab


Welcome! In this hands-on notebook, we will turn the mathematical concepts from our presentation into living, runnable Python code.

---

### What You Will Build Today:

1. The Neural Zoo in Code: Represent membrane voltage, binary spike trains, and multi-neuron population matrices.

2. Brain Vector Physics: Implement dendritic vector addition, attentional gain, dot products, and motor null spaces.

3. Synaptic Matrix Operations: Model synaptic transmission ($y = Wx$) and solve the forward EEG mixing problem.

4. The Muscle Covariance Seesaw: Discover how agonist-antagonist muscles (biceps vs. triceps) produce negative and positive covariance.

5. Eigen-Everything: Uncover the unshakeable resonant modes of a neural network ($Cv = \lambda v$).

6. Untangling Neural Spaghetti with PCA: Build Principal Component Analysis from scratch in 4 lines of NumPy to discover the hidden 2D clockwork inside 100 chaotic neurons.

7. Brain-Computer Interface (BCI) Decoder: Code the linear algorithm ($v_{cursor} = W_{decode} r$) that lets a paralyzed patient steer a robotic cursor with thoughts!

---

In [ ]:
# Import essential scientific packages
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

# Clean plotting styling
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'figure.titlesize': 16,
    'figure.autolayout': True
})

print('Setup complete! Libraries imported successfully.')

## Part 1: The Neural Data Zoo in Code

Let's see how the biological signals we discussed become numerical arrays in Python.

In [ ]:
# ------------------------------------------------------------
# 1.1 Membrane Voltage Trace (1D Analog Vector)
# ------------------------------------------------------------
time_ms = np.linspace(0, 100, 1000)  # 100 milliseconds sampled at 10 kHz

# Simulate resting membrane potential (-70 mV) with small ion noise
vm = -70.0 + 1.5 * np.random.randn(len(time_ms))

# Inject an action potential spike at t = 45 ms
spike_idx = np.logical_and(time_ms >= 40, time_ms <= 55)
vm[spike_idx] += 100.0 * np.exp(-((time_ms[spike_idx] - 45) ** 2) / (2 * 1.5 ** 2))

# ------------------------------------------------------------
# 1.2 Binary Spike Trains & Firing Rates (N Neurons x T Bins)
# ------------------------------------------------------------
n_neurons = 5
n_time_bins = 200  # 200 milliseconds (1 ms bins)

# Generate binary Poisson spike train: mostly 0s, occasional 1s
spike_matrix = (np.random.rand(n_neurons, n_time_bins) < 0.05).astype(int)

# Plotting the biological data zoo
fig, axs = plt.subplots(2, 1, figsize=(10, 6))

# Plot Membrane Voltage
axs[0].plot(time_ms, vm, color='#2563eb', lw=1.5)
axs[0].axhline(-70, color='#94a3b8', linestyle='--', label='Resting Potential (-70 mV)')
axs[0].axhline(-55, color='#dc2626', linestyle=':', label='Threshold (-55 mV)')
axs[0].set_title('Data Type 1: Membrane Voltage (V_m)')
axs[0].set_ylabel('Voltage (mV)')
axs[0].legend(loc='upper right', framealpha=0.9)
axs[0].grid(True, alpha=0.3)

# Plot Spike Raster
for neuron_idx in range(n_neurons):
    spike_times = np.where(spike_matrix[neuron_idx, :] == 1)[0]
    axs[1].vlines(spike_times, neuron_idx + 0.6, neuron_idx + 1.4, color='#0f172a', lw=2)

axs[1].set_title('Data Type 2: Binary Spike Raster Plot (N=5 Neurons)')
axs[1].set_xlabel('Time (ms)')
axs[1].set_ylabel('Neuron ID')
axs[1].set_yticks(range(1, n_neurons + 1))
axs[1].set_ylim(0.5, n_neurons + 0.5)
axs[1].grid(True, alpha=0.3)

plt.show()

In [ ]:
# ------------------------------------------------------------
# 1.3 fMRI (Voxels x Time Points)
# ------------------------------------------------------------

# A tiny "brain": 4 × 4 × 4 voxels
brain = np.random.rand(4, 4, 4)

# Visualize one slice of the brain
plt.imshow(brain[:, :, 2], cmap="viridis")
plt.title("One Slice of an fMRI Brain")
plt.xlabel("Voxel")
plt.ylabel("Voxel")
plt.colorbar(label="BOLD signal")
plt.show()

In [ ]:
# 64 voxels, measured at 100 time points
n_voxels = 64
n_timepoints = 100

# Simulated fMRI data
fmri_data = np.random.randn(n_timepoints, n_voxels)

print("Shape:", fmri_data.shape)

In [ ]:
plt.plot(fmri_data[:, 10])

plt.xlabel("Time")
plt.ylabel("BOLD signal")
plt.title("Data Type 3: BOLD Activity of One fMRI Voxel")
plt.show()

In [ ]:
# Activity of the brain at one moment
brain_state = fmri_data[50, :]

print("Brain state:")
print(brain_state)
print("Dimensions:", brain_state.shape)

In [ ]:
# ------------------------------------------------------------
# 1.3 EEG (Electrodes x Time Points)
# ------------------------------------------------------------

# Recording parameters
sampling_rate = 100       # 100 samples per second
duration = 2              # 2 seconds
time = np.arange(0, duration, 1/sampling_rate)

# 8 EEG electrodes
n_channels = 8

# Simulated EEG data
eeg = np.random.randn(n_channels, len(time))

print("EEG shape:", eeg.shape)

In [ ]:
plt.figure(figsize=(10, 5))

for channel in range(8):
    plt.plot(time, eeg[channel] + channel * 5)

plt.xlabel("Time (seconds)")
plt.ylabel("Electrode")
plt.title("Data Type 4: Simulated EEG Recording")
plt.show()

# Firing Rate: From Spikes to Neural Activity

A neuron communicates by generating **action potentials**, also called **spikes**.

Instead of looking at every spike individually, we can summarize how active a neuron is by calculating its **firing rate**.

## Firing Rate

The firing rate is the number of spikes produced by a neuron per unit of time.

It is measured in **Hertz (Hz)**:

$$
1\text{ Hz} = 1\text{ spike/second}
$$

We estimate the firing rate by counting the number of spikes within a time window.

$$
\boxed{
r(t)=\frac{\text{Spikes}(t,t+\Delta t)}{\Delta t}
}
$$

### Where:

* $r(t)$ = firing rate at time $t$, in **Hz**
* $\Delta t$ = duration of the time window, in **seconds**
* $\text{Spikes}(t,t+\Delta t)$ = number of spikes occurring between $t$ and $t+\Delta t$

---

## Example

Suppose we record the following spike times:

$$
0.12,\;0.18,\;0.21,\;0.35,\;0.42\text{ seconds}
$$

We choose a time window of:

$$
\Delta t = 0.1\text{ s} = 100\text{ ms}
$$

Consider the window from $0.1$ to $0.2$ seconds:

$$
[0.1,\;0.2)
$$

There are **2 spikes** in this window:

$$
0.12,\;0.18
$$

Therefore:

$$
r(0.1)=\frac{2}{0.1}
$$

$$
\boxed{r(0.1)=20\text{ Hz}}
$$

So, during this 100 ms window, the neuron's estimated firing rate is **20 Hz**.

---

## Why Use a Time Window?

Spikes are discrete events:

```
|       |  |          |      |
```

----|-------|--|----------|------|----> Time
spike   spike       spike

We can move a time window across the recording and calculate the firing rate at each position.

At each position:

1. Count the spikes inside the window.
2. Divide by the window duration.
3. Move the window forward.
4. Repeat.

This produces a **time-varying firing-rate signal**.

---

## From One Neuron to a Population

If we record several neurons, each neuron has its own firing rate.

For example, with 4 neurons:

$$
\mathbf{r}(t)=
\begin{bmatrix}
r_1(t)\\
r_2(t)\\
r_3(t)\\
r_4(t)
\end{bmatrix}
$$

This is a **neural activity vector**.

For 128 neurons:

$$
\mathbf{r}(t)=
\begin{bmatrix}
r_1(t)\\
r_2(t)\\
\vdots\\
r_{128}(t)
\end{bmatrix}
$$

At each moment, the entire neural population can therefore be represented by a vector of firing rates.

---

## Key Idea

$$
\boxed{
\text{Spikes}
\rightarrow
\text{Firing Rate}
\rightarrow
\text{Neural Activity Vector}
}
$$

This representation allows us to apply mathematical tools such as:

* Vectors and matrices
* Covariance and correlation
* Principal Component Analysis (PCA)
* Dimensionality reduction
* Neural decoding
* Brain-computer interfaces

> **Firing rate gives us a numerical description of how actively a neuron is firing over time.**


### Exercise 1: Computing Firing Rates in Hertz
Goal: A binary spike train is noisy. Convert the 200 ms spike train above into firing rates in Hertz (Hz) for each neuron.

Hint: Total duration $T = 200\text{ ms} = 0.2\text{ seconds}$. $\text{Rate (Hz)} = \frac{\text{Total Spikes}}{\text{Duration (seconds)}}$.

In [ ]:
# TODO: Calculate the firing rate for each neuron in Hertz (Hz)
duration_sec = 0.200  # 200 ms

# --- WRITE YOUR CODE BELOW ---
total_spikes_per_neuron = np.sum(spike_matrix, axis=1)
rates_hz = total_spikes_per_neuron / duration_sec
# -----------------------------

for i, rate in enumerate(rates_hz):
    print(f'Neuron {i+1}: {rate:.1f} Hz ({total_spikes_per_neuron[i]} spikes in 200 ms)')

print(f'\nPopulation State Vector at this interval:\n v = {rates_hz} Hz')

In [ ]:


# Simulated spike times (seconds)
spikes = np.array([
    0.12, 0.18, 0.21, 0.35, 0.42,
    0.55, 0.61, 0.72, 0.85, 0.91
])

# Sliding window
dt = 0.1  # 100 ms

# Time points at which we calculate firing rate
time = np.arange(0, 1, 0.01)

# Calculate firing rate
firing_rate = []

for t in time:
    count = np.sum((spikes >= t) & (spikes < t + dt))
    rate = count / dt
    firing_rate.append(rate)

# Plot firing rate
plt.plot(time, firing_rate)

plt.xlabel("Time (s)")
plt.ylabel("Firing rate (Hz)")
plt.title("Estimated Firing Rate from Spike Times")
plt.show()

In [ ]:
spikes = np.array([0.12, 0.18, 0.21, 0.35, 0.42])

t = 0.1
dt = 0.1

count = np.sum((spikes >= t) & (spikes < t + dt))

firing_rate = count / dt

print("Spikes in window:", count)
print("Firing rate:", firing_rate, "Hz")

---
## Part 2: Vector Operations in Biological Neurons


- Vector Addition: Dendritic summation merges sight + sound.
- Scalar Multiplication: Attention/Adrenaline scales the volume dial.
- Dot Product & Orthogonality: Checking if two commands align or cancel out.
- Cosine Similarity: Pattern recognition invariant to overall light level.

In [ ]:
# ============================================================
# 2.1 Dendritic Vector Addition: Sight + Sound
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Sight pathway activity
v_sight = np.array([4.0, 6.0])

# Sound pathway activity
v_sound = np.array([2.0, 1.0])

# Superposition at the dendritic tree
v_combined = v_sight + v_sound


# ============================================================
# 2.2 Scalar Multiplication: Attention Gain
# ============================================================

# High-attention state
alpha_dopamine = 1.8

# Drowsy state
alpha_sleepy = 0.4

# Amplified and reduced responses
v_boosted = alpha_dopamine * v_combined
v_dimmed = alpha_sleepy * v_combined


# ============================================================
# Print the Results
# ============================================================

print("Sight vector:       ", v_sight)
print("Sound vector:       ", v_sound)
print("Combined vector:    ", v_combined)
print()
print("Attention ×1.8:     ", v_boosted)
print("Drowsy state ×0.4:  ", v_dimmed)


# ============================================================
# 2.3 Visualize the Vector Operations
# ============================================================

plt.figure(figsize=(8, 8))

# ------------------------------------------------------------
# Sight vector
# ------------------------------------------------------------
plt.quiver(
    0, 0,
    v_sight[0], v_sight[1],
    angles='xy',
    scale_units='xy',
    scale=1,
    color='#2563eb',
    label='Sight [4, 6]'
)

# ------------------------------------------------------------
# Sound vector
# Starts at the tip of the sight vector
# ------------------------------------------------------------
plt.quiver(
    v_sight[0], v_sight[1],
    v_sound[0], v_sound[1],
    angles='xy',
    scale_units='xy',
    scale=1,
    color='#059669',
    label='Sound [2, 1]'
)

# ------------------------------------------------------------
# Combined vector
# ------------------------------------------------------------
plt.quiver(
    0, 0,
    v_combined[0], v_combined[1],
    angles='xy',
    scale_units='xy',
    scale=1,
    color='#dc2626',
    linewidth=2.5,
    label='Combined [6, 7]'
)

# ------------------------------------------------------------
# Attention amplification
# ------------------------------------------------------------
plt.quiver(
    0, 0,
    v_boosted[0], v_boosted[1],
    angles='xy',
    scale_units='xy',
    scale=1,
    color='#7c3aed',
    label='Attention ×1.8 [10.8, 12.6]'
)

# ------------------------------------------------------------
# Drowsiness / reduced gain
# ------------------------------------------------------------
plt.quiver(
    0, 0,
    v_dimmed[0], v_dimmed[1],
    angles='xy',
    scale_units='xy',
    scale=1,
    color='#f59e0b',
    label='Drowsy ×0.4 [2.4, 2.8]'
)


# ============================================================
# Plot Formatting
# ============================================================

plt.xlim(-1, 14)
plt.ylim(-1, 14)

plt.axhline(0, color='gray', linewidth=0.8)
plt.axvline(0, color='gray', linewidth=0.8)

plt.grid(True, alpha=0.3)

plt.xlabel('Channel 1 Activation')
plt.ylabel('Channel 2 Activation')

plt.title('Dendritic Summation and Attentional Gain')

plt.legend(loc='upper left')

plt.gca().set_aspect('equal', adjustable='box')

plt.show()

### 2.3 The Dot Product & Orthogonal Motor Gating

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^N a_i b_i = \|\mathbf{a}\| \|\mathbf{b}\| \cos(\theta)$$
If two motor signals are orthogonal (dot product = 0), they have zero crosstalk.

In [ ]:
# Two candidate motor commands in 2D:
v_reach_left = np.array([1.0, 1.0])
v_reach_right = np.array([1.0, -1.0])

# Compute the dot product using np.dot()
dot_product = np.dot(v_reach_left, v_reach_right)

# Compute norms (lengths)
norm_left = np.linalg.norm(v_reach_left)
norm_right = np.linalg.norm(v_reach_right)

# Compute angle theta in degrees
cos_theta = dot_product / (norm_left * norm_right)
angle_deg = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))

print(f'Dot Product: {dot_product}')
print(f'Angle between commands: {angle_deg:.1f} degrees (ORTHOGONAL / ZERO CROSSTALK!)')

### Exercise 2: Implementing Cosine Similarity
Goal: Write a function cosine_similarity(a, b) and verify that a dim stimulus [2, 1] and a bright stimulus [20, 10] produce a similarity of 1.0 (perfect pattern match).

In [ ]:
def cosine_similarity(a, b):
    """
    Computes cos(theta) = (a . b) / (||a|| * ||b||)
    """
    # --- WRITE YOUR FUNCTION HERE ---
    numerator = np.dot(a, b)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    return numerator / denominator
    # --------------------------------

# Test cases
v_dim_light = np.array([2.0, 1.0])       # Candlelight viewing a picture
v_bright_light = np.array([20.0, 10.0])   # Sunshine viewing the same picture
v_different_image = np.array([1.0, 8.0])  # A totally different picture

sim_identical = cosine_similarity(v_dim_light, v_bright_light)
sim_different = cosine_similarity(v_dim_light, v_different_image)

print(f'Similarity (Dim vs. Bright same image): {sim_identical:.4f} (Expected: 1.0000)')
print(f'Similarity (Dim vs. Different image):    {sim_different:.4f}')
assert np.isclose(sim_identical, 1.0), 'Check your formula! It should equal 1.0 for scaled vectors.'

---
## Part 3: Matrices as Transformations ($y = Wx$)

A matrix is not just a spreadsheet—it is an action. It rotates, stretches, and mixes electrical currents.
- $x$: Input activity vector across $N$ presynaptic neurons
- $W$: Synaptic connection weights matrix ($M \times N$)
- $y = Wx$: Output activity across $M$ postsynaptic neurons

# How Do Multiple Neurons Influence an Output?

### From Synaptic Connections to a Weight Matrix

Imagine **3 sensory neurons** sending information to **2 motor neurons**.

| Sensory Input | Activity |
|---|---:|
| Sensory neuron 1 | 10 |
| Sensory neuron 2 | 2 |
| Sensory neuron 3 | 0.5 |

Each connection has a **weight** that describes how strongly that input influences the motor neuron.

### Synaptic Weight Matrix

$$
W =
\begin{bmatrix}
0.8 & -0.5 & 1.2 \\
-0.2 & 1.5 & 0.1
\end{bmatrix}
$$

**Rows = output neurons**

**Columns = input neurons**

| | Sensory 1 | Sensory 2 | Sensory 3 |
|---|---:|---:|---:|
| **Motor A** | +0.8 | -0.5 | +1.2 |
| **Motor B** | -0.2 | +1.5 | +0.1 |

### How to interpret the weights

- **Positive weight** → contributes positively to the output
- **Negative weight** → contributes negatively to the output
- **Larger magnitude** → stronger influence

> **The matrix stores all the connection strengths between the input and output neurons.**

# Matrix Multiplication Performs the Integration

The sensory activity is represented as a vector:

$$
\mathbf{x} =
\begin{bmatrix}
10 \\
2 \\
0.5
\end{bmatrix}
$$

The neural computation is:

$$
\boxed{\mathbf{y} = W\mathbf{x}}
$$

Therefore:

$$
\begin{bmatrix}
y_A \\
y_B
\end{bmatrix}
=
\begin{bmatrix}
0.8 & -0.5 & 1.2 \\
-0.2 & 1.5 & 0.1
\end{bmatrix}
\begin{bmatrix}
10 \\
2 \\
0.5
\end{bmatrix}
$$

---

### Motor Neuron A

Take the **first row** of the matrix:

$$
y_A =
(0.8)(10)
+(-0.5)(2)
+(1.2)(0.5)
$$

$$
y_A = 8 - 1 + 0.6
$$

$$
\boxed{y_A = 7.6}
$$

---

### Motor Neuron B

Take the **second row**:

$$
y_B =
(-0.2)(10)
+(1.5)(2)
+(0.1)(0.5)
$$

$$
y_B = -2 + 3 + 0.05
$$

$$
\boxed{y_B = 1.05}
$$

---

### Final Output

$$
\boxed{
\mathbf{y} =
\begin{bmatrix}
7.6 \\
1.05
\end{bmatrix}
}
$$

### Dimension Check

$$
\boxed{
(2\times3)(3\times1)=(2\times1)
}
$$

**3 sensory inputs → 2 motor outputs**

> **Matrix multiplication is performing a weighted sum of the inputs for every output neuron.**

In [ ]:
# Example: 3 Sensory input neurons projecting onto 2 Motor output neurons
x_sensory = np.array([10.0, 2.0, 0.5])  # 3 inputs

# Synaptic weight matrix (2 outputs x 3 inputs)
# Row 1 = weights into Motor Neuron A
# Row 2 = weights into Motor Neuron B
W_synaptic = np.array([
    [ 0.8, -0.5,  1.2],   # Neuron A likes input 1, inhibited by 2, excited by 3
    [-0.2,  1.5,  0.1]    # Neuron B excited strongly by input 2
])

# Matrix-vector multiplication using @ operator or np.dot()
y_motor = W_synaptic @ x_sensory

print('Sensory Input x:   ', x_sensory)
print('Synaptic Matrix W:\n', W_synaptic)
print('Motor Output y = Wx:', y_motor)
print(f'Neuron A receives: (0.8*10) + (-0.5*2) + (1.2*0.5) = {y_motor[0]:.2f}')

---
## Part 4: Covariance and the Muscle Seesaw

In the Slide, we discovered the Muscle Seesaw:
- Positive Covariance (+): Two neurons increase their firing together (e.g. co-contraction to lock a joint).
- Negative Covariance (-): One neuron fires high while the other is silenced (e.g. Biceps contracting while Triceps is reciprocally inhibited!).
- Zero Covariance (0): Unrelated cells.

## C = ($\frac1 T) * (X_c @ X_c.T$)

In [ ]:
# ============================================================
# Covariance of Neural Activity
# Biceps + Triceps + Independent Sensory Neuron
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(42)


# ============================================================
# 1. Generate Neural Activity
# ============================================================

T = 100
t_steps = np.linspace(0, 4 * np.pi, T)

# Biceps: increases when the triceps decreases
biceps = (
    20.0
    + 15.0 * np.sin(t_steps)
    + 2.0 * np.random.randn(T)
)

# Triceps: reciprocal / opposing activity
triceps = (
    20.0
    - 15.0 * np.sin(t_steps)
    + 2.0 * np.random.randn(T)
)

# Toe neuron: unrelated activity
toe_neuron = (
    10.0
    + 3.0 * np.random.randn(T)
)


# ============================================================
# 2. Organize Neural Activity into a Matrix
# ============================================================

# Rows    = neurons
# Columns = time points

X_muscles = np.vstack([
    biceps,
    triceps,
    toe_neuron
])

print("Data matrix shape:", X_muscles.shape)
print("3 neurons × 100 time points")


# ============================================================
# 3. Mean-Center the Data
# ============================================================

X_mean = np.mean(
    X_muscles,
    axis=1,
    keepdims=True
)

X_centered = X_muscles - X_mean


# ============================================================
# 4. Compute Covariance Matrix
# ============================================================

C_muscles = (
    1.0 / T
    * (X_centered @ X_centered.T)
)

labels = [
    "Biceps",
    "Triceps",
    "Toe"
]


# ============================================================
# 5. Print Covariance Matrix
# ============================================================

print("\nCovariance Matrix:")
print(np.round(C_muscles, 2))


# ============================================================
# 6. Create Visualization
# ============================================================

fig, axs = plt.subplots(
    1, 2,
    figsize=(15, 6)
)


# ------------------------------------------------------------
# LEFT: Neural Activity Over Time
# ------------------------------------------------------------

axs[0].plot(
    t_steps,
    biceps,
    linewidth=2.5,
    label="Biceps"
)

axs[0].plot(
    t_steps,
    triceps,
    linewidth=2.5,
    label="Triceps"
)

axs[0].plot(
    t_steps,
    toe_neuron,
    linewidth=1.8,
    linestyle="--",
    label="Toe sensory neuron"
)

axs[0].set_title(
    "Neural Activity Over Time",
    fontsize=15,
    fontweight="bold"
)

axs[0].set_xlabel(
    "Time",
    fontsize=12
)

axs[0].set_ylabel(
    "Firing Rate (Hz)",
    fontsize=12
)

axs[0].legend(
    frameon=True
)

axs[0].grid(
    True,
    alpha=0.25
)


# ------------------------------------------------------------
# RIGHT: Covariance Matrix
# ------------------------------------------------------------

# Use symmetric limits so positive and negative covariance
# are visually comparable.

max_cov = np.max(np.abs(C_muscles))

im = axs[1].imshow(
    C_muscles,
    cmap="coolwarm",
    vmin=-max_cov,
    vmax=max_cov
)

fig.colorbar(
    im,
    ax=axs[1],
    fraction=0.046,
    pad=0.04,
    label="Covariance"
)

axs[1].set_xticks(range(3))
axs[1].set_yticks(range(3))

axs[1].set_xticklabels(labels)
axs[1].set_yticklabels(labels)

axs[1].set_title(
    "Covariance Matrix",
    fontsize=15,
    fontweight="bold"
)


# Add covariance values to each cell
for i in range(3):
    for j in range(3):

        value = C_muscles[i, j]

        # Choose text color based on intensity
        text_color = (
            "white"
            if abs(value) > 0.5 * max_cov
            else "black"
        )

        axs[1].text(
            j,
            i,
            f"{value:.1f}",
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
            color=text_color
        )


plt.suptitle(
    "Covariance Reveals Relationships Between Neural Signals",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()

plt.show()


# ============================================================
# 7. Interpret the Important Relationships
# ============================================================

print("\n--- Key Relationships ---")

print(
    f"Biceps ↔ Triceps: "
    f"{C_muscles[0,1]:.2f}"
)

print(
    f"Biceps ↔ Toe:     "
    f"{C_muscles[0,2]:.2f}"
)

print(
    f"Triceps ↔ Toe:    "
    f"{C_muscles[1,2]:.2f}"
)

---
## Part 5: Eigenvalues & Eigenvectors ($Cv = \lambda v$)

The Slide  revealed the Unshakeable Axes:
When a covariance matrix $C$ multiplies an eigenvector $v$, the vector does not rotate or bend—it is only stretched by factor $\lambda$!

Let's compute the eigenvectors of our muscle covariance matrix using np.linalg.eigh().

In [ ]:
# ============================================================
# EIGENDECOMPOSITION OF THE MUSCLE COVARIANCE MATRIX
# ============================================================
#
# We already have:
#
# C_muscles = covariance matrix of
# [Biceps, Triceps, Toe neuron]
#
# Eigendecomposition asks:
#
#       C v = λ v
#
# where:
#   v = eigenvector = a special direction in the data
#   λ = eigenvalue  = how much variance exists along that direction
#
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Compute eigenvalues and eigenvectors
# ------------------------------------------------------------
#
# C_muscles is symmetric because it is a covariance matrix.
# Therefore, np.linalg.eigh() is the appropriate function.

eigenvalues, eigenvectors = np.linalg.eigh(C_muscles)

# eigh() returns eigenvalues from smallest → largest.
# For PCA, we usually want the largest first.

sort_idx = np.argsort(eigenvalues)[::-1]

eigenvalues = eigenvalues[sort_idx]
eigenvectors = eigenvectors[:, sort_idx]

print(eigenvalues)
print("----")
print(eigenvectors)


In [ ]:
# ------------------------------------------------------------
# 2. Calculate variance explained
# ------------------------------------------------------------

total_variance = np.sum(eigenvalues)

variance_explained = (
    eigenvalues / total_variance
) * 100

cumulative_variance = np.cumsum(variance_explained)
print(cumulative_variance)


In [ ]:
# ------------------------------------------------------------
# 3. Display the eigenvalues
# ------------------------------------------------------------

print("=" * 65)
print("EIGENVALUES: HOW MUCH VARIANCE DOES EACH MODE CAPTURE?")
print("=" * 65)

for i, (value, pct, cum_pct) in enumerate(
    zip(eigenvalues, variance_explained, cumulative_variance)
):

    print(
        f"Mode {i+1}: "
        f"λ = {value:8.2f}   | "
        f"Variance = {pct:6.2f}%   | "
        f"Cumulative = {cum_pct:6.2f}%"
    )


In [ ]:
# ------------------------------------------------------------
# 4. Display the eigenvectors
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EIGENVECTORS: THE SPECIAL DIRECTIONS IN NEURAL ACTIVITY")
print("=" * 65)

labels = ["Biceps", "Triceps", "Toe neuron"]

for i in range(len(eigenvalues)):

    print(f"\nMode {i+1}:")

    for label, weight in zip(labels, eigenvectors[:, i]):
        print(f"  {label:10s}: {weight:+.3f}")


In [ ]:
# ------------------------------------------------------------
# 5. Verify the eigenvector equation
# ------------------------------------------------------------
#
# For an eigenvector v:
#
#       C v = λ v
#
# Let's verify this numerically for the first mode.

v1 = eigenvectors[:, 0]
lambda1 = eigenvalues[0]

left_side = C_muscles @ v1
right_side = lambda1 * v1

print("\n" + "=" * 65)
print("VERIFYING THE EIGENVECTOR EQUATION")
print("=" * 65)

print("\nC @ v1:")
print(np.round(left_side, 3))

print("\nλ1 * v1:")
print(np.round(right_side, 3))

print(
    "\nMaximum numerical difference:",
    np.max(np.abs(left_side - right_side))
)

In [ ]:
# ------------------------------------------------------------
# 6. Interpret the first eigenvector
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("INTERPRETATION OF THE FIRST EIGENVECTOR")
print("=" * 65)

print(
    "\nThe first eigenvector represents the direction "
    "of greatest variance in the data."
)

print(
    "\nIts weights tell us how each signal contributes "
    "to this coordinated pattern:"
)

for label, weight in zip(labels, v1):
    print(f"  {label:10s}: {weight:+.3f}")

print(
    "\nIf Biceps and Triceps have opposite signs, "
    "the mode represents an opposing / seesaw pattern."
)

print(
    "\nThe exact sign is arbitrary:"
)

print("    v and -v represent the SAME eigenvector direction.")

print(
    "\nTherefore, focus on the relative signs and magnitudes, "
    "not whether the first number happens to be positive."
)

In [ ]:
# ============================================================
# 7. VISUALIZATION 1: VARIANCE EXPLAINED
# ============================================================

plt.figure(figsize=(8, 5))

modes = np.arange(1, len(eigenvalues) + 1)

plt.bar(modes, variance_explained)

plt.xlabel("Eigenmode")
plt.ylabel("Variance explained (%)")
plt.title("How Much Neural Variance Does Each Eigenmode Explain?")

plt.xticks(modes)

for x, y in zip(modes, variance_explained):
    plt.text(
        x,
        y + 1,
        f"{y:.1f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

---
## Part 6: PCA From Scratch — Untangling the 1,000-Neuron Spaghetti!

In the Slides, we experienced:
- The Crisis: A monkey reaches in two directions (Left vs. Right). We record 100 neurons simultaneously. The raw traces look like an unintelligible plate of spaghetti.
- The PCA Solution:
  1. Center the data: $X_c = X - \mu$
  2. Covariance matrix: $C = \frac{1}{T} X_c X_c^T$
  3. Eigen-decompose: $C v_i = \lambda_i v_i$
  4. Project: $Y = V_k^T X_c$
- The Revelation: The chaotic spaghetti untangles into clean, beautiful 2D rotational orbits!

In [ ]:
# ------------------------------------------------------------
# Step 6.1: Simulate 100 Neurons in Motor Cortex
# ------------------------------------------------------------
N_neurons = 100
T_reach = 80  # 80 time points during the reach
t_axis = np.linspace(0, 1.0, T_reach)

# True underlying 2D rotational dynamics (Churchland et al., 2012)
# Reach Left = counter-clockwise circle
true_mode1_left = np.sin(2 * np.pi * t_axis)
true_mode2_left = np.cos(2 * np.pi * t_axis)

# Reach Right = clockwise circle (inverted phase)
true_mode1_right = -np.sin(2 * np.pi * t_axis)
true_mode2_right = np.cos(2 * np.pi * t_axis)

# Mix these 2 true commands into 100 neurons through random tuning weights
tuning_weights = np.random.randn(N_neurons, 2)

# Raw neural recordings = (Tuning @ Modes) + biological noise
noise_level = 0.6
X_left = tuning_weights @ np.vstack([true_mode1_left, true_mode2_left]) + noise_level * np.random.randn(N_neurons, T_reach)
X_right = tuning_weights @ np.vstack([true_mode1_right, true_mode2_right]) + noise_level * np.random.randn(N_neurons, T_reach)

# Concatenate both reach trials into one dataset (100 neurons x 160 time points)
X_full = np.hstack([X_left, X_right])

# ------------------------------------------------------------
# Step 6.2: View the Raw 100-Neuron Spaghetti Disaster!
# ------------------------------------------------------------
plt.figure(figsize=(10, 4))
for neuron_i in range(N_neurons):
    plt.plot(X_full[neuron_i, :], alpha=0.35, color='#475569')
plt.axvline(T_reach, color='#dc2626', linestyle='--', label='Trial Transition (Left -> Right Reach)')
plt.title('The 100-Neuron Spaghetti Disaster (Raw Firing Rates)')
plt.xlabel('Time Points')
plt.ylabel('Firing Rate (Hz)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.show()

### Step 6.3: Implement PCA from Scratch in 4 Lines!
Let's write the complete PCA algorithm without using scikit-learn.

In [ ]:
# === PCA FROM SCRATCH ===
# 1. Mean-center each neuron's activity
X_mean = np.mean(X_full, axis=1, keepdims=True)
X_centered = X_full - X_mean

# 2. Build the N x N Covariance Matrix
total_T = X_full.shape[1]
C_pop = (1.0 / total_T) * (X_centered @ X_centered.T)

# 3. Eigendecomposition and sorting
eigenvalues, eigenvectors = np.linalg.eigh(C_pop)
sort_idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sort_idx]
eigenvectors = eigenvectors[:, sort_idx]

# 4. Pick top k=2 Principal Components and project the data: Y = V_k.T @ X_centered
V_k = eigenvectors[:, :2]  # Shape: (100 neurons, 2 PCs)
Y_projected = V_k.T @ X_centered  # Shape: (2 PCs, 160 time points)

# Split back into Left and Right reaches
Y_left = Y_projected[:, :T_reach]
Y_right = Y_projected[:, T_reach:]

# Calculate variance explained by top 2 PCs
var_pct = (np.sum(eigenvalues[:2]) / np.sum(eigenvalues)) * 100
print(f'SUCCESS! Top 2 Principal Components capture {var_pct:.1f}% of the total variance across 100 neurons!')

In [ ]:
# ------------------------------------------------------------
# Step 6.4: The Revelation — Neural Trajectories in PCA Space
# ------------------------------------------------------------
#
# We now visualize the neural activity after projecting it
# onto the first two principal components.
#
# Each point represents the state of the neural population
# at one moment in time.
#
# The result is a 2D "neural state space":
#
#       x-axis = PC1
#       y-axis = PC2
#
# If the two movements occupy different regions or follow
# different trajectories, PCA has revealed structure that
# was hidden in the original high-dimensional activity.
# ------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt

# Number of time points
n_left = Y_left.shape[1]
n_right = Y_right.shape[1]

# ------------------------------------------------------------
# 1. Create the figure
# ------------------------------------------------------------

plt.figure(figsize=(9, 8))

# ------------------------------------------------------------
# 2. Plot LEFT reach trajectory
# ------------------------------------------------------------

plt.plot(
    Y_left[0, :],
    Y_left[1, :],
    linewidth=3,
    label="Reach Left"
)

# Starting point
plt.scatter(
    Y_left[0, 0],
    Y_left[1, 0],
    s=120,
    marker="o",
    label="Left: Start"
)

# Ending point
plt.scatter(
    Y_left[0, -1],
    Y_left[1, -1],
    s=180,
    marker="*",
    label="Left: End"
)

# ------------------------------------------------------------
# 3. Plot RIGHT reach trajectory
# ------------------------------------------------------------

plt.plot(
    Y_right[0, :],
    Y_right[1, :],
    linewidth=3,
    label="Reach Right"
)

# Starting point
plt.scatter(
    Y_right[0, 0],
    Y_right[1, 0],
    s=120,
    marker="o",
    label="Right: Start"
)

# Ending point
plt.scatter(
    Y_right[0, -1],
    Y_right[1, -1],
    s=180,
    marker="*",
    label="Right: End"
)

# ------------------------------------------------------------
# 4. Mark the origin
# ------------------------------------------------------------

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

# ------------------------------------------------------------
# 5. Labels and title
# ------------------------------------------------------------

plt.xlabel(
    "Principal Component 1 (PC1)",
    fontsize=12
)

plt.ylabel(
    "Principal Component 2 (PC2)",
    fontsize=12
)

plt.title(
    f"Neural Trajectories in PCA State Space\n"
    f"First Two Principal Components Explain {var_pct:.1f}% of Variance",
    fontsize=14
)

# ------------------------------------------------------------
# 6. Grid and legend
# ------------------------------------------------------------

plt.grid(
    True,
    alpha=0.25
)

plt.legend(
    fontsize=10,
    loc="best"
)

# ------------------------------------------------------------
# 7. Keep the geometry honest
# ------------------------------------------------------------
#
# Equal scaling makes distances and angles in PC1-PC2 space
# visually meaningful.

plt.axis("equal")

plt.tight_layout()
plt.show()

---
## Part 7: Building a Brain-Computer Interface (BCI)

In the Slides, we met Cathy: a paralyzed patient who wanted to move a robotic arm.
The linear velocity decoding algorithm is:
$$\mathbf{v}_{\text{cursor}}(t) = \mathbf{W}_{\text{decode}} \cdot \mathbf{r}(t) + \mathbf{b}_0$$

Let's train a real linear decoder using Ordinary Least Squares (np.linalg.lstsq)!

In [ ]:
# ------------------------------------------------------------
# 7.1 Simulate BCI Training Session
# ------------------------------------------------------------
N_electrodes = 64   # 64-channel Utah array recording motor cortex
T_samples = 400     # 400 calibration time steps

# True 2D cursor velocities the patient was imagining [Vx, Vy]
true_vx = np.sin(np.linspace(0, 6 * np.pi, T_samples))
true_vy = np.cos(np.linspace(0, 6 * np.pi, T_samples))
V_true = np.vstack([true_vx, true_vy])  # Shape: (2, 400)

# Firing rates of 64 electrodes (tuned to direction + noise)
W_biological = np.random.randn(N_electrodes, 2)
R_rates = W_biological @ V_true + 0.5 * np.random.randn(N_electrodes, T_samples)

# ------------------------------------------------------------
# 7.2 Calibrate Decoder Matrix W_decode using Least Squares
# We want: V = W_decode @ R  ==>  R.T @ W_decode.T = V.T
# ------------------------------------------------------------
W_decode_transposed, residuals, rank, s = np.linalg.lstsq(R_rates.T, V_true.T, rcond=None)
W_decode = W_decode_transposed.T   # Shape: (2 velocities, 64 neurons)

print('Decoder Calibrated Successfully!')
print(f'W_decode Matrix Dimensions: {W_decode.shape} (2 velocity outputs x 64 neuron inputs)')

# ------------------------------------------------------------
# 7.3 Test the BCI Online on Brand New Neural Activity!
# ------------------------------------------------------------
# Generate new test signals
t_test = np.linspace(0, 4 * np.pi, 200)
test_vx = np.sin(t_test)
test_vy = np.cos(t_test)
V_test_true = np.vstack([test_vx, test_vy])
R_test_neural = W_biological @ V_test_true + 0.5 * np.random.randn(N_electrodes, len(t_test))

# Real-time BCI decoding: v_decoded = W_decode @ r
V_decoded = W_decode @ R_test_neural

# Calculate cursor positions by integrating velocity: position = cumsum(velocity * dt)
dt = 0.05
pos_true_x = np.cumsum(V_test_true[0, :]) * dt
pos_true_y = np.cumsum(V_test_true[1, :]) * dt

pos_decoded_x = np.cumsum(V_decoded[0, :]) * dt
pos_decoded_y = np.cumsum(V_decoded[1, :]) * dt

# Plot the Result
plt.figure(figsize=(7, 7))
plt.plot(pos_true_x, pos_true_y, color='#059669', lw=3, label='Patient Mental Target (True Path)')
plt.plot(pos_decoded_x, pos_decoded_y, color='#dc2626', linestyle='--', lw=2.5, label='BCI Decoded Cursor Path')
plt.scatter(pos_decoded_x[0], pos_decoded_y[0], color='#0f172a', s=100, label='Start')
plt.scatter(pos_decoded_x[-1], pos_decoded_y[-1], color='#dc2626', s=120, marker='*', label='Goal Reached!')

plt.title('Live Brain-Computer Interface: Mind-Controlled Cursor')
plt.xlabel('Screen X Position')
plt.ylabel('Screen Y Position')
plt.grid(True, alpha=0.3)
plt.legend(loc='upper left')
plt.show()

---
## 🏆 Grand Challenge for Participants

### The Silent Speech Challenge
Suppose we record 128 speech-cortex electrodes while a user silently thinks of three vowel sounds:
- Vowel "Ah" (Vector shape $128 \times 1$)
- Vowel "Ee" (Vector shape $128 \times 1$)
- Vowel "Oo" (Vector shape $128 \times 1$)

Your Mission:
1. Run the starter code below to generate the 3 vowel vectors.
2. Compute the cosine similarity matrix between all 3 pairs.
3. Answer: Which two vowels have the most similar neural patterns in this simulated brain?

In [ ]:
# Generate 3 simulated phoneme patterns in speech motor cortex
np.random.seed(101)
base_mouth_movement = np.random.rand(128)

v_ah = base_mouth_movement * 2.0 + np.random.randn(128) * 0.3
v_ee = base_mouth_movement * 1.8 + np.random.randn(128) * 0.4
v_oo = np.random.rand(128) * 3.0 + np.random.randn(128) * 0.3  # very different vocal posture

# TODO: Calculate the cosine similarities between:
# (1) 'Ah' and 'Ee'
# (2) 'Ah' and 'Oo'
# (3) 'Ee' and 'Oo'

# --- WRITE YOUR CODE BELOW ---
sim_ah_ee = cosine_similarity(v_ah, v_ee)
sim_ah_oo = cosine_similarity(v_ah, v_oo)
sim_ee_oo = cosine_similarity(v_ee, v_oo)
# -----------------------------

print(f"Similarity ('Ah' vs 'Ee'): {sim_ah_ee:.4f}")
print(f"Similarity ('Ah' vs 'Oo'): {sim_ah_oo:.4f}")
print(f"Similarity ('Ee' vs 'Oo'): {sim_ee_oo:.4f}")

print("\nConclusion: 'Ah' and 'Ee' share high similarity because both use open lip postures!")

---
## Summary Checklist

Congratulations on completing the MAMBA Linear Algebra & Neural Data Lab! You now know how to:
- [x] Represent single-cell traces and population arrays in NumPy.
- [x] Add vectors to model synaptic summation and scale them for neuromodulatory gain.
- [x] Use dot products to detect orthogonal motor gating null spaces.
- [x] Build an empirical covariance matrix $C = \frac{1}{T} X_c X_c^T$ and read agonist-antagonist muscle seesaws.
- [x] Decompose $C$ into eigenvalues and eigenvectors.
- [x] Implement PCA from scratch to extract smooth 2D neural trajectories from 100 noisy channels.
- [x] Train a real linear BCI decoder using least squares to steer a cursor.

Keep exploring, and remember: Every eigenvector in your brain tells a story about how your neurons move together!